# Section 3 — Quantization: FP16 vs 4-bit Benchmark
**Model:** Qwen2.5-1.5B-Instruct

**Runtime:** Must use GPU (T4) — Runtime → Change runtime type → T4 GPU

In [ ]:
!pip install -q torch transformers accelerate bitsandbytes huggingface_hub psutil

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem/1024**3:.1f} GB')
else:
    print('WARNING: No GPU. Switch runtime to T4 GPU.')

In [ ]:
import time, gc, psutil
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
MAX_TOKENS = 150

PROMPTS = [
    'Explain insurance underwriting in 2-3 sentences.',
    'Write a Python function to calculate factorial.',
    'What are differences between supervised and unsupervised learning?',
    'List benefits of Mediterranean diet in bullet points.',
    'Translate to French: The weather is beautiful today.',
]

def benchmark(model, tokenizer, label):
    model.eval()
    results = []
    for i, p in enumerate(PROMPTS):
        msgs = [{'role': 'user', 'content': p}]
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors='pt').to(model.device)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=MAX_TOKENS, do_sample=False)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0
        new_toks = out[0][inputs['input_ids'].shape[1]:]
        output = tokenizer.decode(new_toks, skip_special_tokens=True)
        tps = len(new_toks)/elapsed
        results.append({'prompt': p[:40], 'tokens': len(new_toks), 'time': elapsed, 'tok_s': tps, 'output': output})
        print(f'  [{label}] P{i+1}: {len(new_toks)} tok, {elapsed:.2f}s, {tps:.1f} tok/s')
    return results

print('Benchmark function ready.')

In [ ]:
# === FP16 Model ===
print('Loading FP16 model...')
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model_fp16 = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True)

fp16_vram = torch.cuda.memory_allocated()/1024**2
print(f'FP16 VRAM: {fp16_vram:.0f} MB')

print('\n--- FP16 Benchmark ---')
fp16_results = benchmark(model_fp16, tokenizer, 'FP16')

del model_fp16
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# === 4-bit NF4 Model ===
print('Loading 4-bit NF4 model...')
torch.cuda.reset_peak_memory_stats()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model_4bit = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map='auto', trust_remote_code=True)

bit4_vram = torch.cuda.memory_allocated()/1024**2
print(f'4-bit VRAM: {bit4_vram:.0f} MB')

print('\n--- 4-bit NF4 Benchmark ---')
bit4_results = benchmark(model_4bit, tokenizer, '4-bit')

del model_4bit
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# === Results Table ===
fp16_avg_tps = sum(r['tok_s'] for r in fp16_results)/len(fp16_results)
bit4_avg_tps = sum(r['tok_s'] for r in bit4_results)/len(bit4_results)
reduction = (1 - bit4_vram/fp16_vram)*100

print('='*70)
print('  QUANTIZATION TRADE-OFF SUMMARY')
print('='*70)
print(f'\n{"Metric":<25}{"FP16":<20}{"4-bit NF4":<20}')
print('-'*65)
print(f'{"VRAM (MB)":<25}{fp16_vram:<20.0f}{bit4_vram:<20.0f}')
print(f'{"Avg tok/s":<25}{fp16_avg_tps:<20.1f}{bit4_avg_tps:<20.1f}')
print(f'{"VRAM Reduction":<25}{"baseline":<20}{f"{reduction:.0f}%":<20}')
print(f'{"Quality":<25}{"Full":<20}{"~95% of FP16":<20}')

print('\n\n--- Output Comparison (Prompt 1) ---')
print(f'[FP16]: {fp16_results[0]["output"][:200]}')
print(f'\n[4-bit]: {bit4_results[0]["output"][:200]}')